In [2]:
"""
================================================================================
TRAIN NEUROLOGICAL DETECTION MODELS
Run this file first to train all models
================================================================================
"""

import os
import numpy as np
import pandas as pd
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             balanced_accuracy_score)
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    LABELED_RATIO = 0.2
    SSL_MAX_ITER = 10
    
    # File paths
    PARKINSONS_FILE = 'Parkinsson disease.csv'
    ALZHEIMERS_FILE = 'alzheimers_disease_data.csv'
    STROKE_FILE = 'healthcare-dataset-stroke-data.csv'
    
    # Output paths
    MODEL_PARKINSONS = 'models/parkinsons_model.pkl'
    MODEL_ALZHEIMERS = 'models/alzheimers_model.pkl'
    MODEL_STROKE = 'models/stroke_model.pkl'
    
    @classmethod
    def create_directories(cls):
        os.makedirs('models', exist_ok=True)
        os.makedirs('results', exist_ok=True)
        print("✓ Directories created")


# ============================================================================
# PARKINSON'S DETECTOR
# ============================================================================

class ParkinsonsDetector:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.feature_names = None
        
    def load_data(self, filepath):
        print("\n📂 Loading Parkinson's dataset...")
        try:
            df = pd.read_csv(filepath)
            X = df.drop(['name', 'status'], axis=1)
            y = df['status'].values
            self.feature_names = X.columns.tolist()
            print(f"   Shape: {X.shape}")
            print(f"   Class distribution: {np.bincount(y)}")
            X_scaled = self.scaler.fit_transform(X)
            return X_scaled, y
        except Exception as e:
            print(f"   ❌ Error loading data: {e}")
            raise
    
    def train(self, X, y):
        print("\n" + "=" * 60)
        print("TRAINING PARKINSON'S DETECTOR")
        print("=" * 60)
        
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=Config.TEST_SIZE,
                random_state=Config.RANDOM_STATE, stratify=y
            )
            
            X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
                X_train, y_train, test_size=1 - Config.LABELED_RATIO,
                random_state=Config.RANDOM_STATE, stratify=y_train
            )
            
            print(f"\n📊 Data Setup:")
            print(f"   Labeled patients: {len(X_labeled)}")
            print(f"   Unlabeled patients: {len(X_unlabeled)}")
            print(f"   Test patients: {len(X_test)}")
            
            y_labeled_ssl = y_labeled.copy()
            y_unlabeled_ssl = np.full(len(X_unlabeled), -1)
            X_combined = np.vstack([X_labeled, X_unlabeled])
            y_combined = np.concatenate([y_labeled_ssl, y_unlabeled_ssl])
            
            base_model = XGBClassifier(
                n_estimators=80, max_depth=3, learning_rate=0.1,
                random_state=Config.RANDOM_STATE, use_label_encoder=False, eval_metric='logloss'
            )
            
            self.model = SelfTrainingClassifier(
                base_model, threshold=0.80, criterion='threshold',
                max_iter=Config.SSL_MAX_ITER, verbose=False
            )
            
            print("\n🤖 Training with Self-Training...")
            self.model.fit(X_combined, y_combined)
            print(f"   ✅ Complete! Rounds: {self.model.n_iter_}")
            
            y_pred = self.model.predict(X_test)
            y_proba = self.model.predict_proba(X_test)[:, 1]
            
            metrics = {
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred),
                'recall': recall_score(y_test, y_pred),
                'f1': f1_score(y_test, y_pred),
                'auc_roc': roc_auc_score(y_test, y_proba),
                'balanced_acc': balanced_accuracy_score(y_test, y_pred)
            }
            
            print(f"\n📈 Test Results:")
            for k, v in metrics.items():
                print(f"   {k}: {v:.1%}")
            
            self.test_results = {'y_test': y_test, 'y_pred': y_pred, 'y_proba': y_proba}
            return metrics
            
        except Exception as e:
            print(f"   ❌ Error during training: {e}")
            raise
    
    def save(self, filepath):
        try:
            joblib.dump({'model': self.model, 'scaler': self.scaler, 'feature_names': self.feature_names}, filepath)
            print(f"✅ Model saved: {filepath}")
        except Exception as e:
            print(f"   ❌ Error saving model: {e}")
    
    def plot_confusion_matrix(self):
        try:
            cm = confusion_matrix(self.test_results['y_test'], self.test_results['y_pred'])
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=['Healthy', 'Parkinson\'s'],
                        yticklabels=['Healthy', 'Parkinson\'s'])
            plt.title('Parkinson\'s Disease - Confusion Matrix')
            plt.ylabel('Actual')
            plt.xlabel('Predicted')
            plt.savefig('results/parkinsons_confusion_matrix.png', dpi=150, bbox_inches='tight')
            plt.close()
            print("   Saved: results/parkinsons_confusion_matrix.png")
        except Exception as e:
            print(f"   ⚠️ Could not save confusion matrix: {e}")


# ============================================================================
# ALZHEIMER'S DETECTOR
# ============================================================================

class AlzheimerDetector:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.encoders = {}
        self.feature_names = None
        
    def load_data(self, filepath):
        print("\n📂 Loading Alzheimer's dataset...")
        try:
            df = pd.read_csv(filepath)
            
            categorical_cols = df.select_dtypes(include=['object']).columns
            for col in categorical_cols:
                if col not in ['PatientID', 'Diagnosis', 'DoctorInCharge']:
                    self.encoders[col] = LabelEncoder()
                    df[col] = self.encoders[col].fit_transform(df[col].astype(str))
            
            X = df.drop(['PatientID', 'Diagnosis', 'DoctorInCharge'], axis=1)
            y = df['Diagnosis'].values
            self.feature_names = X.columns.tolist()
            
            for col in X.columns:
                if X[col].isnull().sum() > 0:
                    X[col].fillna(X[col].median(), inplace=True)
            
            print(f"   Shape: {X.shape}")
            print(f"   Class distribution: {np.bincount(y)}")
            
            X_scaled = self.scaler.fit_transform(X)
            return X_scaled, y
        except Exception as e:
            print(f"   ❌ Error loading data: {e}")
            raise
    
    def train(self, X, y):
        print("\n" + "=" * 60)
        print("TRAINING ALZHEIMER'S DETECTOR")
        print("=" * 60)
        
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=Config.TEST_SIZE,
                random_state=Config.RANDOM_STATE, stratify=y
            )
            
            X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
                X_train, y_train, test_size=1 - Config.LABELED_RATIO,
                random_state=Config.RANDOM_STATE, stratify=y_train
            )
            
            print(f"\n📊 Data Setup:")
            print(f"   Labeled patients: {len(X_labeled)}")
            print(f"   Unlabeled patients: {len(X_unlabeled)}")
            print(f"   Test patients: {len(X_test)}")
            
            y_labeled_ssl = y_labeled.copy()
            y_unlabeled_ssl = np.full(len(X_unlabeled), -1)
            X_combined = np.vstack([X_labeled, X_unlabeled])
            y_combined = np.concatenate([y_labeled_ssl, y_unlabeled_ssl])
            
            base_model = GradientBoostingClassifier(
                n_estimators=100, max_depth=4, learning_rate=0.1, random_state=Config.RANDOM_STATE
            )
            
            self.model = SelfTrainingClassifier(
                base_model, threshold=0.75, criterion='threshold',
                max_iter=Config.SSL_MAX_ITER, verbose=False
            )
            
            print("\n🤖 Training with Self-Training...")
            self.model.fit(X_combined, y_combined)
            print(f"   ✅ Complete! Rounds: {self.model.n_iter_}")
            
            y_pred = self.model.predict(X_test)
            y_proba = self.model.predict_proba(X_test)[:, 1]
            
            metrics = {
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred),
                'recall': recall_score(y_test, y_pred),
                'f1': f1_score(y_test, y_pred),
                'auc_roc': roc_auc_score(y_test, y_proba),
                'balanced_acc': balanced_accuracy_score(y_test, y_pred)
            }
            
            print(f"\n📈 Test Results:")
            for k, v in metrics.items():
                print(f"   {k}: {v:.1%}")
            
            self.test_results = {'y_test': y_test, 'y_pred': y_pred, 'y_proba': y_proba}
            return metrics
            
        except Exception as e:
            print(f"   ❌ Error during training: {e}")
            raise
    
    def save(self, filepath):
        try:
            joblib.dump({'model': self.model, 'scaler': self.scaler, 'encoders': self.encoders, 'feature_names': self.feature_names}, filepath)
            print(f"✅ Model saved: {filepath}")
        except Exception as e:
            print(f"   ❌ Error saving model: {e}")
    
    def plot_confusion_matrix(self):
        try:
            cm = confusion_matrix(self.test_results['y_test'], self.test_results['y_pred'])
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=['Healthy', 'Alzheimer\'s'],
                        yticklabels=['Healthy', 'Alzheimer\'s'])
            plt.title('Alzheimer\'s Disease - Confusion Matrix')
            plt.ylabel('Actual')
            plt.xlabel('Predicted')
            plt.savefig('results/alzheimers_confusion_matrix.png', dpi=150, bbox_inches='tight')
            plt.close()
            print("   Saved: results/alzheimers_confusion_matrix.png")
        except Exception as e:
            print(f"   ⚠️ Could not save confusion matrix: {e}")


# ============================================================================
# STROKE DETECTOR
# ============================================================================

class StrokeDetector:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.encoders = {}
        self.feature_names = None
        
    def load_data(self, filepath):
        print("\n📂 Loading Stroke dataset...")
        try:
            df = pd.read_csv(filepath)
            X = df.drop(['id', 'stroke'], axis=1)
            y = df['stroke'].values
            self.feature_names = X.columns.tolist()
            
            categorical_cols = X.select_dtypes(include=['object']).columns
            for col in categorical_cols:
                self.encoders[col] = LabelEncoder()
                X[col] = self.encoders[col].fit_transform(X[col].astype(str))
            
            for col in X.columns:
                if X[col].isnull().sum() > 0:
                    X[col].fillna(X[col].median(), inplace=True)
            
            print(f"   Shape: {X.shape}")
            print(f"   Class distribution: {np.bincount(y)}")
            
            X_scaled = self.scaler.fit_transform(X)
            return X_scaled, y
        except Exception as e:
            print(f"   ❌ Error loading data: {e}")
            raise
    
    def train(self, X, y):
        print("\n" + "=" * 60)
        print("TRAINING STROKE DETECTOR")
        print("=" * 60)
        
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=Config.TEST_SIZE,
                random_state=Config.RANDOM_STATE, stratify=y
            )
            
            X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
                X_train, y_train, test_size=1 - Config.LABELED_RATIO,
                random_state=Config.RANDOM_STATE, stratify=y_train
            )
            
            print(f"\n📊 Data Setup:")
            print(f"   Labeled patients: {len(X_labeled)}")
            print(f"   Unlabeled patients: {len(X_unlabeled)}")
            print(f"   Test patients: {len(X_test)}")
            
            healthy_count = np.sum(y_labeled == 0)
            stroke_count = np.sum(y_labeled == 1)
            class_weight = healthy_count / stroke_count if stroke_count > 0 else 19.5
            print(f"   Class imbalance ratio: {healthy_count/stroke_count:.1f}:1")
            print(f"   Using class weight: {class_weight:.1f}")
            
            y_labeled_ssl = y_labeled.copy()
            y_unlabeled_ssl = np.full(len(X_unlabeled), -1)
            X_combined = np.vstack([X_labeled, X_unlabeled])
            y_combined = np.concatenate([y_labeled_ssl, y_unlabeled_ssl])
            
            base_model = XGBClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                scale_pos_weight=class_weight, random_state=Config.RANDOM_STATE,
                use_label_encoder=False, eval_metric='logloss'
            )
            
            self.model = SelfTrainingClassifier(
                base_model, threshold=0.60, criterion='threshold',
                max_iter=Config.SSL_MAX_ITER, verbose=False
            )
            
            print("\n🤖 Training with Self-Training...")
            self.model.fit(X_combined, y_combined)
            print(f"   ✅ Complete! Rounds: {self.model.n_iter_}")
            
            y_pred = self.model.predict(X_test)
            y_proba = self.model.predict_proba(X_test)[:, 1]
            
            metrics = {
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred, zero_division=0),
                'recall': recall_score(y_test, y_pred, zero_division=0),
                'f1': f1_score(y_test, y_pred, zero_division=0),
                'auc_roc': roc_auc_score(y_test, y_proba),
                'balanced_acc': balanced_accuracy_score(y_test, y_pred)
            }
            
            print(f"\n📈 Test Results:")
            for k, v in metrics.items():
                print(f"   {k}: {v:.1%}")
            
            print(f"\n💡 Note: Low precision is expected due to class imbalance.")
            print(f"   The model catches {metrics['recall']:.0%} of stroke cases!")
            
            self.test_results = {'y_test': y_test, 'y_pred': y_pred, 'y_proba': y_proba}
            return metrics
            
        except Exception as e:
            print(f"   ❌ Error during training: {e}")
            raise
    
    def save(self, filepath):
        try:
            joblib.dump({'model': self.model, 'scaler': self.scaler, 'encoders': self.encoders, 'feature_names': self.feature_names}, filepath)
            print(f"✅ Model saved: {filepath}")
        except Exception as e:
            print(f"   ❌ Error saving model: {e}")
    
    def plot_confusion_matrix(self):
        try:
            cm = confusion_matrix(self.test_results['y_test'], self.test_results['y_pred'])
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=['No Stroke', 'Stroke'],
                        yticklabels=['No Stroke', 'Stroke'])
            plt.title('Stroke Risk - Confusion Matrix')
            plt.ylabel('Actual')
            plt.xlabel('Predicted')
            plt.savefig('results/stroke_confusion_matrix.png', dpi=150, bbox_inches='tight')
            plt.close()
            print("   Saved: results/stroke_confusion_matrix.png")
        except Exception as e:
            print(f"   ⚠️ Could not save confusion matrix: {e}")


# ============================================================================
# MAIN TRAINING FUNCTION
# ============================================================================

def main():
    print("=" * 70)
    print("🧠 NEUROLOGICAL DETECTION MODELS")
    print("Semi-Supervised Learning | Only 20% Labeled Data Needed")
    print("=" * 70)
    
    Config.create_directories()
    
    results = {}
    
    # 1. Train Parkinson's
    print("\n" + "=" * 70)
    print("1. PARKINSON'S DISEASE DETECTOR")
    print("=" * 70)
    
    try:
        parkinson = ParkinsonsDetector()
        X_park, y_park = parkinson.load_data(Config.PARKINSONS_FILE)
        results['Parkinson\'s'] = parkinson.train(X_park, y_park)
        parkinson.save(Config.MODEL_PARKINSONS)
        parkinson.plot_confusion_matrix()
    except Exception as e:
        print(f"❌ Parkinson's model training failed: {e}")
    
    # 2. Train Alzheimer's
    print("\n" + "=" * 70)
    print("2. ALZHEIMER'S DISEASE DETECTOR")
    print("=" * 70)
    
    try:
        alzheimer = AlzheimerDetector()
        X_alz, y_alz = alzheimer.load_data(Config.ALZHEIMERS_FILE)
        results['Alzheimer\'s'] = alzheimer.train(X_alz, y_alz)
        alzheimer.save(Config.MODEL_ALZHEIMERS)
        alzheimer.plot_confusion_matrix()
    except Exception as e:
        print(f"❌ Alzheimer's model training failed: {e}")
    
    # 3. Train Stroke
    print("\n" + "=" * 70)
    print("3. STROKE RISK DETECTOR")
    print("=" * 70)
    
    try:
        stroke = StrokeDetector()
        X_stroke, y_stroke = stroke.load_data(Config.STROKE_FILE)
        results['Stroke'] = stroke.train(X_stroke, y_stroke)
        stroke.save(Config.MODEL_STROKE)
        stroke.plot_confusion_matrix()
    except Exception as e:
        print(f"❌ Stroke model training failed: {e}")
    
    # Final Summary
    if results:
        print("\n" + "=" * 70)
        print("📊 FINAL RESULTS SUMMARY")
        print("=" * 70)
        
        summary_df = pd.DataFrame(results).T.round(4)
        print("\n", summary_df)
    
    print("\n" + "=" * 70)
    print("✅ TRAINING COMPLETE")
    print("=" * 70)


if __name__ == "__main__":
    main()

🧠 NEUROLOGICAL DETECTION MODELS
Semi-Supervised Learning | Only 20% Labeled Data Needed
✓ Directories created

1. PARKINSON'S DISEASE DETECTOR

📂 Loading Parkinson's dataset...
   Shape: (195, 22)
   Class distribution: [ 48 147]

TRAINING PARKINSON'S DETECTOR

📊 Data Setup:
   Labeled patients: 31
   Unlabeled patients: 125
   Test patients: 39

🤖 Training with Self-Training...
   ✅ Complete! Rounds: 5

📈 Test Results:
   accuracy: 79.5%
   precision: 83.9%
   recall: 89.7%
   f1: 86.7%
   auc_roc: 85.3%
   balanced_acc: 69.8%
✅ Model saved: models/parkinsons_model.pkl
   Saved: results/parkinsons_confusion_matrix.png

2. ALZHEIMER'S DISEASE DETECTOR

📂 Loading Alzheimer's dataset...
   Shape: (2149, 32)
   Class distribution: [1389  760]

TRAINING ALZHEIMER'S DETECTOR

📊 Data Setup:
   Labeled patients: 343
   Unlabeled patients: 1376
   Test patients: 430

🤖 Training with Self-Training...
   ✅ Complete! Rounds: 8

📈 Test Results:
   accuracy: 91.4%
   precision: 92.6%
   recall: 82.

In [1]:
"""
================================================================================
🧠 NEUROLOGICAL DETECTION MODELS (COMPLETE VERSION WITH IMBALANCE HANDLING)
================================================================================
"""

import os
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, auc, classification_report)

from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

class Config:
    TEST_SIZE = 0.2
    LABELED_RATIO = 0.3  # Increased for better performance
    RANDOM_STATE = 42
    USE_SMOTE = True  # Enable SMOTE for imbalanced datasets

    PARKINSONS_FILE = 'Parkinsson disease.csv'
    ALZHEIMERS_FILE = 'alzheimers_disease_data.csv'
    STROKE_FILE = 'healthcare-dataset-stroke-data.csv'

    @staticmethod
    def create_dirs():
        os.makedirs("results", exist_ok=True)


# ============================================================
# 📊 GRAPH FUNCTION
# ============================================================

def generate_all_plots(model, feature_names, X_test, y_test, y_pred, y_proba, metrics, name):
    
    # Print classification report
    print(f"\n📋 {name} - Classification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # 1. Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.heatmap(cm, annot=True, fmt='d', ax=ax1, cmap='Blues')
    ax1.set_title(f"{name} - Confusion Matrix (Counts)")
    ax1.set_xlabel('Predicted')
    ax1.set_ylabel('Actual')
    
    sns.heatmap(cm_norm, annot=True, fmt='.2%', ax=ax2, cmap='Blues')
    ax2.set_title(f"{name} - Confusion Matrix (Normalized)")
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('Actual')
    
    plt.tight_layout()
    plt.savefig(f"results/{name.lower()}_cm.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})", linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.5)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{name} - ROC Curve')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.savefig(f"results/{name.lower()}_roc.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 3. Performance Chart
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    all_metrics = {**metrics, 'specificity': specificity}
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(list(all_metrics.keys()), list(all_metrics.values()), 
                   color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c'])
    plt.ylim(0, 1)
    plt.title(f"{name} - Performance Metrics", fontsize=14, fontweight='bold')
    plt.ylabel('Score', fontsize=12)
    plt.xticks(rotation=45)
    
    for bar, value in zip(bars, all_metrics.values()):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{value:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f"results/{name.lower()}_performance.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 4. Feature Importance
    try:
        trained_model = model.estimator_
    except:
        trained_model = model
    
    if hasattr(trained_model, "feature_importances_"):
        importances = trained_model.feature_importances_
        
        # Top 15 features
        indices = np.argsort(importances)[::-1][:15]
        
        plt.figure(figsize=(10, 8))
        plt.barh(range(len(indices)), importances[indices][::-1])
        plt.yticks(range(len(indices)), [feature_names[i] for i in indices[::-1]])
        plt.title(f"{name} - Top 15 Feature Importance", fontsize=14, fontweight='bold')
        plt.xlabel("Importance", fontsize=12)
        plt.tight_layout()
        plt.savefig(f"results/{name.lower()}_features.png", dpi=300, bbox_inches='tight')
        plt.close()
    else:
        print(f"⚠️ Feature importance not available for {name}")


# ============================================================
# 🧠 TRAIN FUNCTION
# ============================================================

def train_model(X, y, feature_names, base_model, name, use_smote=False):
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=Config.TEST_SIZE,
        stratify=y, random_state=Config.RANDOM_STATE
    )
    
    print(f"\n🎯 {name} - Class Distribution:")
    print(f"  Training - Class 0: {sum(y_train==0)}, Class 1: {sum(y_train==1)}")
    print(f"  Test     - Class 0: {sum(y_test==0)}, Class 1: {sum(y_test==1)}")
    
    # Apply SMOTE if requested
    if use_smote and Config.USE_SMOTE:
        print(f"  🔄 Applying SMOTE oversampling...")
        smote = SMOTE(random_state=Config.RANDOM_STATE)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print(f"  After SMOTE - Class 0: {sum(y_train==0)}, Class 1: {sum(y_train==1)}")
    
    # Semi-supervised learning
    X_l, X_u, y_l, _ = train_test_split(
        X_train, y_train,
        test_size=1 - Config.LABELED_RATIO,
        stratify=y_train,
        random_state=Config.RANDOM_STATE
    )
    
    y_u = np.full(len(X_u), -1)
    
    X_combined = np.vstack([X_l, X_u])
    y_combined = np.concatenate([y_l, y_u])
    
    # Train
    model = SelfTrainingClassifier(base_model, verbose=False)
    model.fit(X_combined, y_combined)
    
    # Predict
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc': roc_auc_score(y_test, y_proba)
    }
    
    generate_all_plots(
        model,
        feature_names,
        X_test,
        y_test,
        y_pred,
        y_proba,
        metrics,
        name
    )
    
    return model, metrics


# ============================================================
# 📂 DATA LOADERS
# ============================================================

def load_parkinsons():
    df = pd.read_csv(Config.PARKINSONS_FILE)
    print(f"📁 Parkinson's dataset: {df.shape}")
    
    X = df.drop(['name', 'status'], axis=1)
    y = df['status']
    
    feature_names = X.columns.tolist()
    X_scaled = StandardScaler().fit_transform(X)
    
    return X_scaled, y, feature_names


def load_alzheimers():
    df = pd.read_csv(Config.ALZHEIMERS_FILE)
    print(f"📁 Alzheimer's dataset: {df.shape}")
    
    for col in df.select_dtypes(include='object'):
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    
    df = df.fillna(df.median(numeric_only=True))
    
    X = df.drop(['PatientID', 'Diagnosis', 'DoctorInCharge'], axis=1)
    y = df['Diagnosis']
    
    feature_names = X.columns.tolist()
    X_scaled = StandardScaler().fit_transform(X)
    
    return X_scaled, y, feature_names


def load_stroke():
    df = pd.read_csv(Config.STROKE_FILE)
    print(f"📁 Stroke dataset: {df.shape}")
    
    for col in df.select_dtypes(include='object'):
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    
    df = df.fillna(df.median(numeric_only=True))
    
    X = df.drop(['id', 'stroke'], axis=1)
    y = df['stroke']
    
    stroke_ratio = sum(y) / len(y)
    print(f"  ⚠️ Stroke prevalence: {stroke_ratio:.2%}")
    
    feature_names = X.columns.tolist()
    X_scaled = StandardScaler().fit_transform(X)
    
    return X_scaled, y, feature_names


# ============================================================
# 🚀 MAIN
# ============================================================

def main():
    
    Config.create_dirs()
    results = {}
    
    print("="*80)
    print("🧠 NEUROLOGICAL DISEASE DETECTION SYSTEM")
    print("="*80)
    
    # Parkinson's
    print("\n" + "="*40)
    print("🎯 TRAINING PARKINSON'S MODEL")
    print("="*40)
    X, y, cols = load_parkinsons()
    model, metrics = train_model(
        X, y, cols,
        XGBClassifier(eval_metric='logloss', random_state=Config.RANDOM_STATE),
        "Parkinsons",
        use_smote=False
    )
    results['Parkinsons'] = metrics
    
    # Alzheimer's
    print("\n" + "="*40)
    print("🎯 TRAINING ALZHEIMER'S MODEL")
    print("="*40)
    X, y, cols = load_alzheimers()
    model, metrics = train_model(
        X, y, cols,
        GradientBoostingClassifier(random_state=Config.RANDOM_STATE),
        "Alzheimers",
        use_smote=False
    )
    results['Alzheimers'] = metrics
    
    # Stroke (with SMOTE)
    print("\n" + "="*40)
    print("🎯 TRAINING STROKE MODEL (WITH SMOTE)")
    print("="*40)
    X, y, cols = load_stroke()
    model, metrics = train_model(
        X, y, cols,
        XGBClassifier(eval_metric='logloss', scale_pos_weight=5, random_state=Config.RANDOM_STATE),
        "Stroke",
        use_smote=True
    )
    results['Stroke'] = metrics
    
    # Final Results
    print("\n" + "="*80)
    print("📊 FINAL RESULTS")
    print("="*80)
    
    results_df = pd.DataFrame(results).T
    print("\n", results_df.round(4))
    
    # Save to CSV
    results_df.to_csv("results/model_performance_summary.csv")
    print("\n✅ Results saved to 'results/model_performance_summary.csv'")
    
    # Comparison Plot
    plt.figure(figsize=(12, 6))
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    
    for i, metric in enumerate(metrics_to_plot):
        plt.subplot(2, 3, i+1)
        values = [results[model][metric] for model in results.keys()]
        bars = plt.bar(results.keys(), values, color=['#3498db', '#2ecc71', '#e74c3c'])
        plt.title(f'{metric.upper()}', fontweight='bold')
        plt.ylim(0, 1)
        plt.xticks(rotation=45)
        for bar, v in zip(bars, values):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{v:.3f}', ha='center', fontsize=9)
    
    plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig("results/all_models_comparison.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    print("\n🎉 All plots saved to 'results/' directory!")


if __name__ == "__main__":
    main()

🧠 NEUROLOGICAL DISEASE DETECTION SYSTEM

🎯 TRAINING PARKINSON'S MODEL
📁 Parkinson's dataset: (195, 24)

🎯 Parkinsons - Class Distribution:
  Training - Class 0: 38, Class 1: 118
  Test     - Class 0: 10, Class 1: 29

📋 Parkinsons - Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.30      0.43        10
           1       0.80      0.97      0.88        29

    accuracy                           0.79        39
   macro avg       0.78      0.63      0.65        39
weighted avg       0.79      0.79      0.76        39


🎯 TRAINING ALZHEIMER'S MODEL
📁 Alzheimer's dataset: (2149, 35)

🎯 Alzheimers - Class Distribution:
  Training - Class 0: 1111, Class 1: 608
  Test     - Class 0: 278, Class 1: 152

📋 Alzheimers - Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.97      0.94       278
           1       0.93      0.84      0.88       152

    accuracy                      